In [ ]:
##################################################################
# # ! Icosphere
##################################################################
# %%
# # ! Setup
import pyvista as pv;
import numpy as np;
import pandas as pd;
import time;
from gravity_forward_numpy import gsphere;
from gravity_forward_numba import VecWerSch_numba;

In [ ]:
# # ! Icosphere
xc = 0.1; yc = 0.0; zc = 2.0;
a = 1.5;
rho   = 1000.;
xgv = np.linspace(-10., 10., 101);
ygv = np.linspace(-10., 10., 101);
z0 = 0.;
[X2d, Y2d] = np.meshgrid(xgv, ygv);
Z2d = z0 * np.ones(X2d.shape);
XPs, YPs, ZPs = map(lambda x: x.flatten(), [X2d, Y2d, Z2d]);
P = np.column_stack((XPs, YPs, ZPs));

vol = 4*np.pi*a**3/3; area = 4*np.pi*a**2;
print(f'Sphere volume: {vol:12.6f}', flush=True);
print(f'Sphere area: {area:12.6f}', flush=True);
Ns = np.arange(9);
for N in Ns:
    icosph = pv.Icosphere(radius=a, center=(xc, yc, zc), nsub=N);
    Vert  = icosph.points;
    Faces = icosph.regular_faces;
    print(f'N : {N}');
    print(f'Number of <Vertex> : {Vert.shape[0]}');
    print(f'Number of <Faces>  : {Faces.shape[0]}');
    re_vol = np.abs(icosph.volume-vol)/vol * 100;
    re_area = np.abs(icosph.area-area)/area * 100;
    print(f'Icosphere volume: {icosph.volume:12.6f}, e={re_vol:.4f}%', flush=True);
    print(f'Icosphere area: {icosph.area:12.6f}, e={re_area:.4f}%', flush=True);
######## * Sphere
t1 = time.time();
V, gx, gy, gz, Txx, Tyy, Tzz, Txy, Txz, Tyz \
    = gsphere(P[:,0], P[:,1], P[:,2], xc, yc, zc, a, rho);
tc_sphere = time.time() - t1;
print(f'<Sphere> time cost: {tc_sphere:.2f} sec');
######## * Polyhedron
t1 = time.time();
V_cal, gx_cal, gy_cal, gz_cal, \
Txx_cal, Tyy_cal, Tzz_cal, Txy_cal, Txz_cal, Tyz_cal \
    = VecWerSch_numba(P, Vert, Faces, rho);
tc_numba = time.time() - t1;
print(f'<Polyhedron> time cost: {tc_numba:.2f} sec');

Sphere volume:    14.137167
Sphere area:    28.274334
N : 0
Number of <Vertex> : 12
Number of <Faces>  : 20
Icosphere volume:     8.559510, e=39.4539%
Icosphere area:    21.542721, e=23.8082%
Sphere area:    28.274334
N : 0
Number of <Vertex> : 12
Number of <Faces>  : 20
Icosphere volume:     8.559510, e=39.4539%
Icosphere area:    21.542721, e=23.8082%
N : 1
Number of <Vertex> : 42
Number of <Faces>  : 80
Icosphere volume:    12.348155, e=12.6547%
Icosphere area:    26.248347, e=7.1655%
N : 2
Number of <Vertex> : 162
Number of <Faces>  : 320
Icosphere volume:    13.658643, e=3.3849%
Icosphere area:    27.740391, e=1.8884%
N : 3
Number of <Vertex> : 642
Number of <Faces>  : 1280
Icosphere volume:    14.015311, e=0.8620%
Icosphere area:    28.138894, e=0.4790%
N : 4
Number of <Vertex> : 2562
Number of <Faces>  : 5120
Icosphere volume:    14.106560, e=0.2165%
Icosphere area:    28.240349, e=0.1202%
N : 5
Number of <Vertex> : 10242
Number of <Faces>  : 20480
Icosphere volume:    14.129506

In [ ]:
# ! #  Pandas disp stats 
fields = ['V', 'gx', 'gy', 'gz', 'Txx', 'Tyy', 'Tzz', 'Txy', 'Txz', 'Tyz']

df_ref = pd.DataFrame({
    name: {'Min': arr.min(), 'Max': arr.max(), 'Mean': arr.mean(), 'Std': arr.std()}
    for name, arr in zip(fields, [V, gx, gy, gz, Txx, Tyy, Tzz, Txy, Txz, Tyz])
}).T
df_cal = pd.DataFrame({
    name: {'Min': arr.min(), 'Max': arr.max(), 'Mean': arr.mean(), 'Std': arr.std()}
    for name, arr in zip(fields, [V_cal, gx_cal, gy_cal, gz_cal, 
                                  Txx_cal, Tyy_cal, Tzz_cal, Txy_cal, Txz_cal, Tyz_cal])
}).T
df_diff = df_cal - df_ref

def print_table(df, title):
    print(f"\n{title}")
    print("=" * len(title))
    print(df.to_string(float_format="{:12.6f}".format))
print_table(df_ref, "Reference")
print_table(df_cal, "Polyhedron")
print_table(df_diff, "Difference")


Reference
             Min          Max         Mean          Std
V       0.065739     0.471190     0.138188     0.069492
gx     -9.058147     9.058147     0.006171     2.005550
gy     -9.055963     9.055963     0.000000     2.005571
gz      0.063821    23.500740     1.198026     2.669170
Txx  -116.624622    23.843904    -0.617182    10.944427
Tyy  -117.503702    23.750439    -0.617076    10.944441
Tzz    -4.219713   234.128325     1.234258    17.877373
Txy   -32.764925    32.764925     0.000000     6.323560
Txz  -100.511959   100.511959     0.002922    12.674168
Tyz  -100.768558   100.768558    -0.000000    12.674169

Polyhedron
             Min          Max         Mean          Std
V       0.065739     0.471186     0.138187     0.069492
gx     -9.058069     9.058069     0.006171     2.005533
gy     -9.055887     9.055887     0.000000     2.005554
gz      0.063821    23.500540     1.198016     2.669147
Txx  -116.623665    23.843703    -0.617177    10.944335
Tyy  -117.502628    23.75

In [ ]:
df_ref

,Min,Max,Mean,Std
V,0.065739,0.471190,1.381883e-01,0.069492
gx,-9.058147,9.058147,6.171263e-03,2.005550
gy,-9.055963,9.055963,8.915741e-17,2.005571
gz,0.063821,23.500740,1.198026e+00,2.669170
Txx,-116.624622,23.843904,-6.171822e-01,10.944427
Tyy,-117.503702,23.750439,-6.170757e-01,10.944441
Tzz,-4.219713,234.128325,1.234258e+00,17.877373
Txy,-32.764925,32.764925,5.572338e-18,6.323560
Txz,-100.511959,100.511959,2.921893e-03,12.674168
Tyz,-100.768558,100.768558,-3.566296e-16,12.674169


In [ ]:
df_cal

,Min,Max,Mean,Std
V,0.065739,0.471186,1.381871e-01,0.069492
gx,-9.058069,9.058069,6.171211e-03,2.005533
gy,-9.055887,9.055887,2.050620e-15,2.005554
gz,0.063821,23.500540,1.198016e+00,2.669147
Txx,-116.623665,23.843703,-6.171770e-01,10.944335
Tyy,-117.502628,23.750240,-6.170704e-01,10.944348
Tzz,-4.219677,234.126292,1.234247e+00,17.877221
Txy,-32.764648,32.764648,-1.988628e-15,6.323506
Txz,-100.511076,100.511076,2.921868e-03,12.674060
Tyz,-100.767733,100.767733,2.853037e-15,12.674061


In [ ]:
df_diff

,Min,Max,Mean,Std
V,-5.568659e-07,-0.000004,-1.170572e-06,-5.886668e-07
gx,7.712050e-05,-0.000077,-5.227576e-08,-1.698750e-05
gy,7.599375e-05,-0.000076,1.961463e-15,-1.699556e-05
gz,-5.406188e-07,-0.000200,-1.014829e-05,-2.261423e-05
Txx,9.575662e-04,-0.000202,5.228049e-06,-9.250859e-05
Tyy,1.074822e-03,-0.000199,5.227147e-06,-9.316136e-05
Tzz,3.574630e-05,-0.002032,-1.045520e-05,-1.516816e-04
Txy,2.778412e-04,-0.000278,-1.994200e-15,-5.369374e-05
Txz,8.828047e-04,-0.000883,-2.475086e-08,-1.072520e-04
Tyz,8.250947e-04,-0.000825,3.209667e-15,-1.078156e-04
